In [8]:
from pathlib import Path
import anndata
import numpy as np
import scanpy as sc
import scvi
import seaborn as sns
import torch
import matplotlib.pyplot as plt
from cytetype import CyteType

In [9]:
scvi.settings.seed = 0
print("Last run with scvi-tools version:", scvi.__version__)

Seed set to 0


Last run with scvi-tools version: 1.4.2


In [10]:
sc.set_figure_params(figsize=(6, 6), frameon=False)
sns.set_theme()
torch.set_float32_matmul_precision("high")
model_dir = Path.cwd().parent / "models"

%config InlineBackend.print_figure_kwargs={"facecolor": "w"}
%config InlineBackend.figure_format="retina"

# Load data

In [11]:
adata = sc.read(
    filename=Path.cwd().parent / "data" / "breastcancer_scatlas.h5ad",
    backup_url=(
        "https://datasets.cellxgene.cziscience.com/7cdea341-ca7a-40fd-8192-b8ecb2d7b91e.h5ad"
    ),
)
adata

AnnData object with n_obs × n_vars = 621200 × 37389
    obs: 'tissue_ontology_term_id', 'tissue_type', 'assay_ontology_term_id', 'disease_ontology_term_id', 'cell_type_ontology_term_id', 'self_reported_ethnicity_ontology_term_id', 'development_stage_ontology_term_id', 'sex_ontology_term_id', 'donor_id', 'suspension_type', 'grade', 'author_cell_type', 'batch', 'is_primary_data', 'cell_type', 'assay', 'disease', 'sex', 'tissue', 'self_reported_ethnicity', 'development_stage', 'observation_joinid'
    var: 'feature_is_filtered', 'feature_name', 'feature_reference', 'feature_biotype', 'feature_length', 'feature_type'
    uns: 'batch_condition', 'citation', 'default_embedding', 'organism', 'organism_ontology_term_id', 'schema_reference', 'schema_version', 'title'
    obsm: 'X_rpca', 'X_umap'

In [32]:
# Filter cells with few genes
sc.pp.filter_cells(adata, min_genes=500)

# Filter genes that appear in few cells
sc.pp.filter_genes(adata, min_cells=60)

In [33]:
# Save raw counts to `counts` layer in anndata object
adata.layers["counts"] = adata.X.astype(np.int32)

# Normalize cells to 10,000 counts
sc.pp.normalize_total(adata, target_sum=1e4)

# Change to log scale
sc.pp.log1p(adata)

# Save raw data
adata.raw = adata

In [19]:
# Perform feature selection
sc.pp.highly_variable_genes(
    adata,
    n_top_genes=1500,
    subset=True,
    layer="counts",
    flavor="seurat_v3",
    batch_key="batch",
)

/var/folders/rj/58krw3q53619qt2rbck9wq740000gn/T/ipykernel_91517/2109877985.py:2: UserWarning: `flavor='seurat_v3'` expects raw count data, but non-integers were found.
  sc.pp.highly_variable_genes(


In [43]:
# Setup anndata object for model fitting
scvi.model.SCVI.setup_anndata(
    adata, layer="counts", categorical_covariate_keys=["donor_id", "assay"]
)

# Process data

# Create, train, and save scVI model

In [39]:
type(model_dir)

pathlib.PosixPath

In [44]:
# Instantiate model
model = scvi.model.SCVI(adata)

# Train model
model.train(train_size=0.8, check_val_every_n_epoch=10)

# Save model
model.save(model_dir / "scvi1.pt", overwrite=True)

/Users/otodreas/Desktop/Work/Nygen/Interview/.venv/lib/python3.12/site-packages/scvi/train/_trainrunner.py:98: UserWarning: `accelerator` has been automatically set to `cpu` although 'mps' exists. If you wish to run on mps backend, use explicitly accelerator='mps' in train function.In future releases it will become default for mps supported machines.
  accelerator, lightning_devices, device = parse_device_args(
GPU available: True (mps), used: False
TPU available: False, using: 0 TPU cores
/Users/otodreas/Desktop/Work/Nygen/Interview/.venv/lib/python3.12/site-packages/lightning/pytorch/trainer/setup.py:175: GPU available but not used. You can set it by doing `Trainer(accelerator='gpu')`.
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
/Users/otodreas/Desktop/Work/Nygen/Interview/.venv/lib/python3.12/site-pa

Epoch 1/16:   0%|          | 0/16 [00:00<?, ?it/s]


Detected KeyboardInterrupt, attempting graceful shutdown ...


Exception raised during training. <class 'NameError'> 1
Epoch 1/16:   0%|          | 0/16 [03:30<?, ?it/s]


SystemExit: 1

/Users/otodreas/Desktop/Work/Nygen/Interview/.venv/lib/python3.12/site-packages/IPython/core/interactiveshell.py:3709: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)


In [ ]:
# Load model
model = scvi.model.SCVI.load(Path.cwd().parent / "models" / "scvi1" / "model.pt")

INFO     No backup URL provided for missing file                                                                   
         /home/inf-33-2025/Work/Nygen/Interview/models/scvi1/model.pt/model.pt                                     


NotADirectoryError: [Errno 20] Not a directory: '/home/inf-33-2025/Work/Nygen/Interview/models/scvi1/model.pt/model.pt'

# Plot model training diagnostics

In [4]:
# Assign model history to variable
hist = model.history


# Set matplotlib aesthetics
def plot_metric(train_key, val_key, title, ylabel):
    plt.figure(figsize=(6, 4))
    plt.plot(hist[train_key], label="train")
    plt.plot(hist[val_key], label="validation")
    plt.xlabel("Epoch")
    plt.ylabel(ylabel)
    plt.title(title)
    plt.legend()
    plt.tight_layout()
    plt.show()

NameError: name 'model' is not defined

In [ ]:
# Negative ELBO (Evidence lower bound)
if "elbo_train" in hist:
    plot_metric("elbo_train", "elbo_validation", "ELBO", "ELBO")

In [ ]:
# Reconstruction loss
plot_metric(
    "reconstruction_loss_train",
    "reconstruction_loss_validation",
    "Reconstruction loss",
    "Negative log likelihood",
)

In [ ]:
# Local KL divergence
plot_metric(
    "kl_local_train",
    "kl_local_validation",
    "KL divergence (latent z)",
    "KL(q(z|x)||p(z))",
)

# Cluster

In [ ]:
# TODO: leidenclustering

# Run CyteType

In [ ]:
group_key = "clusters"
annotator = CyteType(
    adata, group_key=group_key, rank_key="rank_genes_" + group_key, n_top_genes=100
)
adata = annotator.run(study_context="Human PBMC from healthy donor")
sc.pl.umap(adata, color="cytetype_annotation_clusters")